# Lindsay Butcher and Lance Prince fetal mouse liver macrophage cell populations

## Cluster and characterize single cell RNASeq data using Seurat

* Art Nasamran <cnasamran@ucsd.edu>
* Based on upstream analysis by Art Nasamran, CCBB <cnasamran@ucsd.edu>

## Table of Contents
* [Background](#Background)
* [Load packages](#Load-packages)
* [Load data](#Load-data)
* [Preprocessing](#Preprocessing)
* [Quality Control](#QC)
* [Normalization](#Normalize-the-data)
* [Identify variable genes](#Identify-variable-features)
* [Scaling](#Scaling-the-data)
* [Dimensional reduction](#Linear-dimensional-reduction)
* [Determine dimensionality of the dataset](#Determine-the-dimensionality-of-the-dataset)
* [Clustering](#Cluster-the-cells)
* [UMAP/tSNE](#Run-non-linear-dimensional-reduction-(UMAP/tSNE))
* [Find differentially expressed genes between clusters](#Find-differentially-expressed-genes-(cluster-biomarkers))
* [Figures](#Figures)
* [Citations](#Citations)
* [R Session Info](#R-Session-Info)

## Background
The count data analyzed in this notebook were produced by the upstream analysis of Art Nasamran of CCBB, who received raw sequencing data and performed alignment and quantification of reads using the `count` function from 10X Genomics [Cell Ranger software](https://support.10xgenomics.com/single-cell-gene-expression/software/pipelines/latest/what-is-cell-ranger).
* Experimental goal(s): Cluster and characterize immune cells (CD45+) from fetal mouse liver from 2 samples.
    * KA_E15_Liver_01 and KA_E15_Liver_02





Methods for this notebook are heavily based upon the [Seurat - Guided Clustering Tutorial (v3)](https://satijalab.org/seurat/v3.0/pbmc3k_tutorial.html) from Satija Lab. Some of the method descriptions are direct quotes from the vignette.

E-mail from Lindsay 7/12/19 with background publications:
    
    "Hi Art,
    Below are some links to articles that may helpful for our 10X data.  While we sequenced all immune cells (Cd45+) from the fetal mouse liver, this project is focused on the macrophage populations so the papers below focus on those populations.
    The libraries went on the sequencer at the beginning of the month so I anticipate having data relatively soon. Please let me know if you have any questions.
    Thanks,
    Lindsay
    UCSD, Dept. of Pediatrics
    Lance Prince Lab
    ldbutcher@ucsd.edu
    858.534.2289

    Specification of tissue-resident macrophages during organogenesis
https://science.sciencemag.org/content/353/6304/aaf4238
    PMID: 27492475 PMCID: PMC5066309 DOI: 10.1126/science.aaf4238

    Single cell RNA sequencing identifies unique inflammatory airspace macrophage subsets
https://insight.jci.org/articles/view/126556
    PMID: 30721157 PMCID: PMC6483508 DOI: 10.1172/jci.insight.126556
    -focus on time point 0 (non-infected subset)

    Single cell RNA sequencing of human liver reveals distinct intrahepatic macrophage populations
https://www.nature.com/articles/s41467-018-06318-7
    PMID: 30348985 PMCID: PMC6197289 DOI: 10.1038/s41467-018-06318-7
    -See cluster Cluster 10; Fig. 2f, two distinct populations of macrophages in the liver

    RNA sequencing and transcriptomal analysis of human monocyte to macrophage differentiation,
https://www.sciencedirect.com/science/article/pii/S0378111913001911?via%3Dihub
    PMID: 23458880 PMCID: PMC3666862 DOI: 10.1016/j.gene.2013.02.015"
    
E-mail from Lance 8/7/19 with analysis and figure requests:
    
    "Art,

    Attached is a spreadsheet that tries to combine and identify the clusters.

    We feel pretty confident that clusters 0,1,6,12,16, and 20 represent neutrophils.
    Clusters 9 and 19 are dendritic cells.
    Clusters 2,3,4,8,10,11, and 15 are macrophage/monocyte populations. We are actually interested in the differences between them so not sure we would want to combine at this time.

    Sheet 2 has 3 lists of genes that we would like to examine further:
    We would like to see the expression of each gene plotted on the UMAP clustering figure.
    We would like to compare violin plots of each gene only on the macrophage/monocyte clusters-2,3,4,8,10,11, and 15.
 
    Does that sound doable? Hopefully not too many genes, but we really aren’t sure exactly what to expect. If this is something we could do through a Jupyter notebook or web based tool, please let us know.

    Thanks again for all of your terrific help!!!!
    Lance"

E-mail from Lindsay 8/28/19 with an analysis request:

    "Hi Art,

    Great.  I was worried that it would too biased to proceed that way so I’m happy that we can try.  We are interested in re-clustering and analyzing with the top DE genes heat map and the GOI heat map, umap, and violin plots for clusters 2, 4, 8, 10, 11, 17.  This would exclude the 2 macrophage clusters on the left of the group (3 & 15).  Please let me know if you have any questions.

    Thanks,
    Lindsay"
    
E-mail from Linsday 9/28/19:

    "Hi Art,

    Lance and I were discussing the macrophage results this week and planning what to do next for the paper.  We noticed that the shapes are somewhat different from the last clustering compared to the one before that that did not save.  Can you run the cluster analysis again so can confirm one of the shapes?

    Thanks,
    Lindsay"

[Table of Contents](#Table-of-Contents)

# Load packages

In [1]:
# install.packages('Seurat') #re-installed on 5/20/19 v2.3.4 to v3
library(Seurat)
library(dplyr)
library(ggplot2)

ERROR: Error in library(Seurat): there is no package called ‘Seurat’


In [3]:
# Load required library
library(remotes)

# Check available versions on CRAN
available_versions <- remotes:::CRAN_package_db()$results[grepl("SeuratObject", remotes:::CRAN_package_db()$results$Package),]
print(available_versions)

# If the version you want is available, install it
# remotes::install_version("SeuratObject", "your_version_here", repos = "https://cran.r-project.org")


Warning message:
“package ‘remotes’ was built under R version 4.2.3”


ERROR: Error in eval(expr, envir, enclos): object 'CRAN_package_db' not found


In [2]:
remotes::install_version("SeuratObject", "3.4.0", repos = c("https://satijalab.r-universe.dev", getOption("repos")))
remotes::install_version("Seurat", "3.4.0", repos = c("https://satijalab.r-universe.dev", getOption("repos")))

Trying https://satijalab.r-universe.dev

Trying https://cran.r-project.org



ERROR: Error in download_version_url(package, version, repos, type): version '3.4.0' is invalid for package 'SeuratObject'


[Table of Contents](#Table-of-Contents)

# Load data

In [ ]:
data_dir1 <- "/Users/sshome/Desktop/E15_liver_analysis_sayane_Nov_2021/10X_processed_files/liver1/filtered_feature_bc_matrix/"
data_dir2 <- "/Users/sshome/Desktop/E15_liver_analysis_sayane_Nov_2021/10X_processed_files/liver2/filtered_feature_bc_matrix/"
cells1.data <- Read10X(data.dir = data_dir1)
cells2.data <- Read10X(data.dir = data_dir2)

[Table of Contents](#Table-of-Contents)

# Preprocessing

## Filter cells and create Seurat objects
* min.cells = include genes with detected expression in at least this many cells (~0.1% of the data)
* min.genes = include cells where at least this many genes are detected

In [ ]:
min_cells <- 0.001
min_genes <- 200

In [ ]:
cells1 <- CreateSeuratObject(counts = cells1.data, 
                             min.cells = round(ncol(cells1.data)*min_cells), 
                             min.features = min_genes,
                             project = "KA_E15_Liver_01")

cells2 <- CreateSeuratObject(counts = cells2.data,
                             min.cells = round(ncol(cells2.data)*min_cells),
                             min.features = min_genes,
                             project = "KA_E15_Liver_02")

# Merge data
* "KA_E15_Liver_01" will be called S1
* "KA_E15_Liver_02" will be called S2

In [ ]:
cells <- merge(cells1, y = cells2, add.cell.ids = c("S1", "S2"), project = "KA_E15_Liver")

In [ ]:
cells
head(colnames(cells))
table(cells$orig.ident)

[Table of Contents](#Table-of-Contents)

# QC
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4758103/ - Single cell QC metrics commonly used in research outlined by Ilicic et al. 2016.

As summarized by Satija lab in their tutorial: 

* "The number of unique genes detected in each cell.
    * Low-quality cells or empty droplets will often have very few genes
    * Cell doublets or multiplets may exhibit an aberrantly high gene count
* Similarly, the total number of molecules detected within a cell (correlates strongly with unique genes)
* The percentage of reads that map to the mitochondrial genome
    * Low-quality / dying cells often exhibit extensive mitochondrial contamination"


## Calculate and store mitochondrial gene percentages

<div class="alert alert-warning">
    <strong>Analyst Note:</strong><br/>
    
* Change pattern = to "^MT-" if your study uses human samples and "^mt-" for mouse.

</div>

In [ ]:
# Mitochondrial gene function (Seurat v3)
calcMitoGenes <- function(seuratObject){
    seuratObject[["percent.mito"]] <- PercentageFeatureSet(object = seuratObject, pattern = "^mt-") #"MT" for human, "mt" for mouse
    return(seuratObject)
}

In [ ]:
cells <- calcMitoGenes(cells)

## Filtering the data
* Remove outliers based on unique gene counts (nGene) and percent mitochondrial genes (percent.mito) for each set of cells.

<div class="alert alert-warning">
    <strong>Analyst Note:</strong><br/>
    
* Look for outliers in the violin plots below. Apply appropriate filters using `subset`.

</div>

In [ ]:
VlnPlot(object = cells, features = c("nFeature_RNA", "percent.mito"), ncol = 2)

In [ ]:
print("No filtering:")
cells
cells <- subset(cells, subset = nFeature_RNA > 700 & nFeature_RNA < 4000 & percent.mito < 10)
print("After filtering:")
cells

In [ ]:
plot1 <- FeatureScatter(cells, feature1 = "nCount_RNA", feature2 = "percent.mito")
plot2 <- FeatureScatter(cells, feature1 = "nCount_RNA", feature2 = "nFeature_RNA")
plot1
plot2

[Table of Contents](#Table-of-Contents)
# Normalize the data
* "The `LogNormalize` method normalizes the feature expression measurements for each cell by the total expression, multiplies this by a scale factor (10,000 by default), and log-transforms the result." - Satija Lab vignette.

In [ ]:
cells <- NormalizeData(cells)

[Table of Contents](#Table-of-Contents)

# Identify variable features
* Seurat v3 implements a new method for variable feature selection based on a variance stabilizing transformation (vst). The `FindVariableFeatures` function calculates a subset of features that exhibit high cell-to-cell variation in the dataset. More details on focusing on variable features in single-cell datasets can be found in this publication: [Account for technical noise in single-cell RNA-seq experiments](https://www.nature.com/articles/nmeth.2645). The details of the new Seurat method can be found in this pre-publication [Comprehensive integration of single cell data](https://www.biorxiv.org/content/biorxiv/early/2018/11/02/460147.full.pdf)

* From seurat documentation: "vst: First, fits a line to the relationship of log(variance) and log(mean) using local polynomial regression (loess). Then standardizes the feature values using the observed mean and expected variance (given by the fitted line). Feature variance is then calculated on the standardized values after clipping to a maximum (see clip.max parameter)."

In [ ]:
cells <- FindVariableFeatures(cells, selection.method = "vst", verbose = FALSE)

In [ ]:
# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(cells), 10)

In [ ]:
# plot variable features with and without labels
plot3 <- VariableFeaturePlot(cells)
plot4 <- LabelPoints(plot = plot3, points = top10, repel = TRUE)
plot3
plot4

[Table of Contents](#Table-of-Contents)

# Scaling the data
* Apply a linear transformation (scaling) that shifts the expression of each gene so that the mean expression across cells is 0. Scaling also transforms the variance across cells to 1, which helps give equal weight to lowly and highly expressed genes in downstream analyses.

In [ ]:
all.genes <- rownames(cells)

In [ ]:
cells <- ScaleData(cells, features = all.genes)

[Table of Contents](#Table-of-Contents)

# Linear dimensional reduction
* Reduce the dimensionality of the data using PCA on the scaled data.

In [ ]:
cells <- RunPCA(cells, features = VariableFeatures(object = cells))

In [ ]:
# Visualize dimension loadings
VizDimLoadings(cells, dims = 1:2, reduction = "pca")

In [ ]:
DimPlot(cells, reduction = "pca")

[Table of Contents](#Table-of-Contents)

# Determine the dimensionality of the dataset
* "To overcome the extensive technical noise in any single feature for scRNA-seq data, Seurat clusters cells based on their PCA scores, with each PC essentially representing a ‘metafeature’ that combines information across a correlated feature set. The top principal components therefore represent a robust compression of the dataset.

* In [Macosko et al](https://www.cell.com/abstract/S0092-8674(15)00549-8), we implemented a resampling test inspired by the JackStraw procedure. We randomly permute a subset of the data (1% by default) and rerun PCA, constructing a ‘null distribution’ of feature scores, and repeat this procedure. We identify ‘significant’ PCs as those who have a strong enrichment of low p-value features." -Satija Lab

* 2 methods can be used to determine the number of PCs to use in downstream analyses -- jackstraw plot and elbow plot

## Jackstraw plot

In [ ]:
cells <- JackStraw(cells, num.replicate = 100)
cells <- ScoreJackStraw(cells, dims = 1:20)

In [ ]:
JackStrawPlot(cells, dims = 1:20)

## Elbow plot
* The elbow plot shows the PCs based on the percentage of variance explained by each one. Typically, we look for the area in which including additional PCs provides diminishing returns. In an ideal situation, this would appear as an elbow or joint where the plot flattens out.

In [ ]:
ElbowPlot(cells)

### Notes:
* The jackstraw plot provides dubious results as there were no clear dropoffs in p-value in dimensions 1 through 20.
* The elbow plot suggests we can choose PCs 16-18 as a cutoff point.
* For this analysis, I will choose 16 as the cutoff.

In [ ]:
# Set dimension cutoff
dims <- 16

[Table of Contents](#Table-of-Contents)

# Cluster the cells
Quoted from the vignette:

"Seurat v3 applies a graph-based clustering approach, building upon initial strategies in ([Macosko et al](https://www.cell.com/abstract/S0092-8674(15)00549-8)). Importantly, the distance metric which drives the clustering analysis (based on previously identified PCs) remains the same. However, our approach to partioning the cellular distance matrix into clusters has dramatically improved. Our approach was heavily inspired by recent manuscripts which applied graph-based clustering approaches to scRNA-seq data [SNN-Cliq, Xu and Su, Bioinformatics, 2015](https://academic.oup.com/bioinformatics/article/31/12/1974/214505) and CyTOF data [PhenoGraph, Levine et al., Cell, 2015](http://www.ncbi.nlm.nih.gov/pubmed/26095251). Briefly, these methods embed cells in a graph structure - for example a K-nearest neighbor (KNN) graph, with edges drawn between cells with similar feature expression patterns, and then attempt to partition this graph into highly interconnected ‘quasi-cliques’ or ‘communities’.

As in PhenoGraph, we first construct a KNN graph based on the euclidean distance in PCA space, and refine the edge weights between any two cells based on the shared overlap in their local neighborhoods (Jaccard similarity). This step is performed using the `FindNeighbors` function, and takes as input the previously defined dimensionality of the dataset.

To cluster the cells, we next apply modularity optimization techniques such as the Louvain algorithm (default) or SLM [SLM, Blondel et al., Journal of Statistical Mechanics](http://dx.doi.org/10.1088/1742-5468/2008/10/P10008), to iteratively group cells together, with the goal of optimizing the standard modularity function. The `FindClusters` function implements this procedure, and contains a resolution parameter that sets the ‘granularity’ of the downstream clustering, with increased values leading to a greater number of clusters. We find that setting this parameter between 0.4-1.2 typically returns good results for single-cell datasets of around 3K cells. Optimal resolution often increases for larger datasets. The clusters can be found using the `Idents` function."

In [ ]:
# Set seed for reproducibility
seed <- 7

In [ ]:
cells <- FindNeighbors(cells, dims = 1:dims)

In [ ]:
cells <- FindClusters(cells, 
                      resolution = c(0.9), 
                      random.seed = seed)

In [ ]:
cells <- RunTSNE(cells, dims = 1:dims, seed.use = seed)

In [ ]:
head(cells@meta.data)

In [ ]:
table(Idents(cells))

[Table of Contents](#Table-of-Contents)

# Run non-linear dimensional reduction (UMAP/tSNE)
"Seurat offers several non-linear dimensional reduction techniques, such as tSNE and UMAP, to visualize and explore these datasets. The goal of these algorithms is to learn the underlying manifold of the data in order to place similar cells together in low-dimensional space. Cells within the graph-based clusters determined above should co-localize on these dimension reduction plots. As input to the UMAP and tSNE, we suggest using the same PCs as input to the clustering analysis." -Satija Lab

## An argument for UMAP over tSNE
Becht et al. 2019 compared uniform manifold approximation and projection (UMAP) vs. t-SNE applications in single-cell RNASeq and mass cytometry data in their publication, ["Dimensionality reduction for visualizing single-cell data using UMAP"](https://www.nature.com/articles/nbt.4314?WT.feed_name=subjects_data-mining). As summarized in their abstract, they found, "that UMAP provides the fastest run times, highest reproducibility and the most meaningful organization of cell clusters". Due to recent advancements in dimensionality reduction algorithms, UMAP was also added to Seurat v3.

In [ ]:
cells <- RunUMAP(cells, dims = 1:dims)

In [ ]:
DimPlot(cells, reduction = "umap", group.by = "orig.ident")

In [ ]:
#DimPlot(cells, reduction = "umap", group.by = "RNA_snn_res.0.5")

In [ ]:
#DimPlot(cells, reduction = "umap", group.by = "RNA_snn_res.0.7", label = TRUE)

In [ ]:
DimPlot(cells, reduction = "umap", group.by = "RNA_snn_res.0.9", label = TRUE)

In [ ]:
#DimPlot(cells, reduction = "umap", group.by = "RNA_snn_res.1.1", label = TRUE)

## FeaturePlots by nCount_RNA, nFeature_RNA, and percent.mito
* Make sure that the clusters aren't driven by any of these variables

In [ ]:
FeaturePlot(cells, features = c("nCount_RNA"), pt.size = 0.5)

In [ ]:
FeaturePlot(cells, features = c("nFeature_RNA"), pt.size = 0.5)

In [ ]:
FeaturePlot(cells, features = c("percent.mito"), pt.size = 0.5)

### Select a clustering resolution
* 8/5/19: after meeting with Lance and Lindsay - try 0.9 resolution - they're most interested in looking at macrophage populations in the bottom right corner of the UMAP plots

In [ ]:
Idents(cells) <- "RNA_snn_res.0.9"

In [ ]:
table(Idents(cells))

In [ ]:
table(cells@meta.data$orig.ident, cells@meta.data$`RNA_snn_res.0.9`)

In [ ]:
?DimPlot

In [ ]:
# Visualization
DimPlot(cells, reduction = "umap", label = TRUE)

In [ ]:
DimPlot(cells, reduction = "tsne", group.by = "RNA_snn_res.0.9", label = TRUE)

In [ ]:
head(Idents(cells), 5)

In [ ]:
# Save cells object
#save(cells, file = "KA_E15_Liver_combined_Dec15.Rdata")

In [ ]:
# Load cells object when continuing
load("KA_E15_Liver_combined.Rdata")

In [ ]:
View(cells)

[Table of Contents](#Table-of-Contents)

# Find differentially expressed genes (cluster biomarkers)
"Seurat can help you find markers that define clusters via differential expression. By default, it identifes positive and negative markers of a single cluster (specified in ident.1), compared to all other cells. `FindAllMarkers` automates this process for all clusters, but you can also test groups of clusters vs. each other, or against all cells.

The min.pct argument requires a gene to be detected at a minimum percentage in either of the two groups of cells, and the thresh.test argument requires a gene to be differentially expressed (on average) by some amount between the two groups. You can set both of these to 0, but with a dramatic increase in time - since this will test a large number of genes that are unlikely to be highly discriminatory. As another option to speed up these computations, max.cells.per.ident can be set. This will downsample each identity class to have no more cells than whatever this is set to. While there is generally going to be a loss in power, the speed increases can be significiant and the most highly differentially expressed genes will likely still rise to the top." -Satija Lab

<div class="alert alert-warning">
    <strong>Analyst Note:</strong><br/>
    
* Set n_clust to match the number of clusters in the resolution you chose. Remember, numbering starts at 0.

Seurat supports the following differential expression tests:
* "wilcox" : Wilcoxon rank sum test (default)
* "bimod" : Likelihood-ratio test for single cell feature expression, (McDavid et al., Bioinformatics, 2013)
* "roc" : Standard AUC classifier
* "t" : Student’s t-test
* "poisson" : Likelihood ratio test assuming an underlying negative binomial distribution. Use only for UMI-based datasets
* "negbinom" : Likelihood ratio test assuming an underlying negative binomial distribution. Use only for UMI-based datasets
* "LR" : Uses a logistic regression framework to determine differentially expressed genes. Constructs a logistic regression model predicting group membership based on each feature individually and compares this to a null model with a likelihood ratio test.
* "MAST" : GLM-framework that treates cellular detection rate as a covariate (Finak et al, Genome Biology, 2015) (Installation instructions)
* "DESeq2" : DE based on a model using the negative binomial distribution (Love et al, Genome Biology, 2014) (Installation instructions)

For MAST and DESeq2 please ensure that these packages are installed separately in order to use them as part of Seurat. Once installed, use the test.use parameter can be used to specify which DE test to use.
</div>

In [ ]:
FindAllMarkers(cells, ident.1, ident.2 = NULL, genes.use = NULL,
  thresh.use = 0.25, test.use = "bimod", min.pct = 0.1,
  min.diff.pct = 0.05, print.bar = TRUE, only.pos = FALSE,
  max.cells.per.ident = Inf, return.thresh = 0.01, do.print = FALSE,
  random.seed = 1)

In [ ]:
# Change cell identities to annotated names - assign to column name from metadata
Idents(cells) <- "annot_clusters"

In [ ]:
Idents(cells)

In [ ]:
# Number of clusters
#n_clust <- c(0:25)
# Use cluster names for annotated lists
n_clust <- unique(Idents(cells))

In [ ]:
n_clust

In [ ]:
getwd()

In [ ]:
setwd("")

In [ ]:
# min.pct - only test genes that are detected in a minimum fraction of min.pct cells in either of 2 populations.
#         - meant to speed up the function by not testing genes that are infrequently expressed
# 8/8/19 - reran after combining the clusters
for (i in n_clust){
    name <- FindMarkers(cells, ident.1 = i, print.bar = FALSE)
    write.csv(name, file = paste("KA_E15_Liver_Cluster", i, "Gene_Markers.csv", sep = "_"), quote = FALSE, row.names = TRUE)
}

In [ ]:
# find markers for every cluster compared to all remaining cells, report only the positive ones
cells.markers <- FindAllMarkers(cells, only.pos = TRUE, min.pct = 0.05, logfc.threshold = 0.25)

In [ ]:
# Top 2 defining genes for each cluster
cells.markers %>% group_by(cluster) %>% top_n(n = 2, wt = avg_logFC)

## Look for clusters that may need to be combined or split
Compare and contrast defining gene lists between clusters based on a cluster tree. Combine clusters that share defining genes. Consult with researchers/literature to see if the combinations make sense.

In [ ]:
cells <- BuildClusterTree(cells, reorder = FALSE, reorder.numeric = FALSE)

In [ ]:
Tool(object = cells, slot = "BuildClusterTree")

In [ ]:
# After combining clusters 8/8/19
PlotClusterTree(object = cells)

<div class="alert alert-warning">
    <strong>Analyst Note:</strong><br/>
    
* Check the dendrogram above and find clusters that may need to be combined. Compare DE gene lists and ask the researcher(s) if the markers are similar enough to merge clusters or if they potentially represent subtypes and should be kept separate.

</div>

## Reassign clusters
* Merge clusters 9 and 19 -> Dendritic Cells
* Merge clusters 0, 1, 6, 12, 16, and 20 -> Neutrophils
* Clusters 2, 3, 4, 8, 10, 11, 15, and 17 are macrophage/monocyte populations that will remain separated
* Cluster 21 -> NK cells
* Cluster 22 -> HSCs
* Cluster 7 -> B cells

In [ ]:
# Create new variable for manual cluster ID
cells@meta.data$annot_clusters <- as.character(cells@meta.data$RNA_snn_res.0.9)

In [ ]:
table(cells@meta.data$RNA_snn_res.0.9)

In [ ]:
# Change 9 and 19 to "Dendritic cells" (n=897)
cells@meta.data$annot_clusters[which(cells@meta.data$annot_clusters == "9")] <- rep("Dendritic cells", length(which(cells@meta.data$annot_clusters == "9")))
cells@meta.data$annot_clusters[which(cells@meta.data$annot_clusters == "19")] <- rep("Dendritic cells", length(which(cells@meta.data$annot_clusters == "19")))


In [ ]:
# Change 0, 1, 6, 12, 16, 20 to "Neutrophils" (n=4169)
cells@meta.data$annot_clusters[which(cells@meta.data$annot_clusters %in% c("0", "1", "6", "12", "16", "20"))] <- rep("Neutrophils", length(which(cells@meta.data$annot_clusters %in% c("0", "1", "6", "12", "16", "20"))))

In [ ]:
# Change 21 to "NK cells" (n=97)
cells@meta.data$annot_clusters[which(cells@meta.data$annot_clusters == "21")] <- rep("NK cells", length(which(cells@meta.data$annot_clusters == "21")))


In [ ]:
# Change 22 to "HSCs" (n=95)
cells@meta.data$annot_clusters[which(cells@meta.data$annot_clusters == "22")] <- rep("HSCs", length(which(cells@meta.data$annot_clusters == "22")))


In [ ]:
# Change 7 to "B cells" (n=755)
cells@meta.data$annot_clusters[which(cells@meta.data$annot_clusters == "7")] <- rep("B cells", length(which(cells@meta.data$annot_clusters == "7")))


In [ ]:
table(cells@meta.data$annot_clusters)

In [ ]:
# cells@meta.data$annot_clusters <- gsub("4", "3", cells@meta.data$annot_clusters)
# cells@meta.data$annot_clusters <- gsub("7", "3", cells@meta.data$annot_clusters)

# Subset and cluster macrophage cells
* Focus on clusters 2, 4, 8, 10, 11, and 17
* Produce lists of DE genes, heatmaps, and violin plots

Seurat developer, leonfodoulian recommends: 
    
    "...[S]imply pass the cluster ID of the cells you'd like to re-cluster to the ident.use argument of Seurat:: SubsetData(), and create a new object with that. Then, you have to re-compute variable genes, scale the data, reduce the dimension and run clustering (pretty much exactly as what you did during the initial step). Once you get your clusters, you can then re-set the identity of the cells from each of the sub-clusters using the Seurat::SetIdent() function." 
Reference: https://github.com/satijalab/seurat/issues/409


### Subset macrophages
* clusters 2, 4, 8, 10, 11, and 17 in RNA_snn_res.0.9
* Rerun on 9/30/19

In [ ]:
Idents(cells) <- "RNA_snn_res.0.9"
cells.macro <- subset(cells, idents = c("2", "4", "8", "10", "11", "17"))

In [ ]:
table(cells.macro@meta.data$RNA_snn_res.0.9)

In [ ]:
# Save cell names as an RData object
head(cells.macro@meta.data)
macronames <- rownames(cells.macro@meta.data)
# save(macronames, file = "macrophage_cellnames.RData")

### Recompute variable genes

In [ ]:
cells.macro

In [ ]:
cells.macro <- FindVariableFeatures(cells.macro, selection.method = "vst", nfeatures = 2000, verbose = FALSE)

In [ ]:
top10 <- head(VariableFeatures(cells.macro), 10)

In [ ]:
# plot variable features with and without labels
plot3 <- VariableFeaturePlot(cells.macro)
plot4 <- LabelPoints(plot = plot3, points = top10, repel = TRUE)
plot3
plot4

### Scale the data

In [ ]:
macro.all.genes <- rownames(cells.macro)

In [ ]:
cells.macro <- ScaleData(cells.macro, features = macro.all.genes)

### Reduce and determine the dimensionality

In [ ]:
cells.macro <- RunPCA(cells.macro, features = VariableFeatures(object = cells.macro))

In [ ]:
# Visualize dimension loadings
VizDimLoadings(cells.macro, dims = 1:2, reduction = "pca")

In [ ]:
DimPlot(cells.macro, reduction = "pca")

In [ ]:
cells.macro <- JackStraw(cells.macro, num.replicate = 100)
cells.macro <- ScoreJackStraw(cells.macro, dims = 1:20)

In [ ]:
# JackStrawPlot(cells.macro, dims = 1:20)

In [ ]:
ElbowPlot(cells.macro)

In [ ]:
# Set dimension cutoff
dims <- 13

### Cluster the macrophages
* Cluster the subset of macrophages using the same methods as above

In [ ]:
# Set seed for reproducibility
seed <- 7

In [ ]:
?FindClusters

In [ ]:
cells.macro <- FindNeighbors(cells.macro, dims = 1:dims)

In [ ]:
cells.macro <- FindClusters(cells.macro, 
                      resolution = c(0.5, 0.7, 0.9, 1.1), 
                      random.seed = seed)

In [ ]:
head(cells.macro@meta.data)

In [ ]:
table(Idents(cells.macro))

In [ ]:
cells.macro <- RunUMAP(cells.macro, dims = 1:dims)

In [ ]:
DimPlot(cells.macro, reduction = "umap", group.by = "orig.ident")

In [ ]:
DimPlot(cells.macro, reduction = "umap", group.by = "RNA_snn_res.0.5", label = TRUE)

In [ ]:
DimPlot(cells.macro, reduction = "umap", group.by = "RNA_snn_res.0.7", label = TRUE)

In [ ]:
DimPlot(cells.macro, reduction = "umap", group.by = "RNA_snn_res.0.9", label = TRUE)

In [ ]:
DimPlot(cells.macro, reduction = "umap", group.by = "RNA_snn_res.1.1", label = TRUE)

In [ ]:
FeaturePlot(cells.macro, features = c("nCount_RNA"), pt.size = 0.5)

In [ ]:
FeaturePlot(cells.macro, features = c("nFeature_RNA"), pt.size = 0.5)

In [ ]:
FeaturePlot(cells.macro, features = c("percent.mito"), pt.size = 0.5)

In [ ]:
# Choose a resolution
Idents(cells.macro) <- "RNA_snn_res.0.7"

In [ ]:
table(cells.macro@meta.data$orig.ident, Idents(cells.macro))

In [ ]:
n_clust <- c(0:9)

In [ ]:
# setwd("20190912_macrophage_cluster_markers/")
#setwd("20190930_macrophage_cluster_markers/")
# Find DE genes by cluster
for (i in n_clust){
    name <- FindMarkers(cells.macro, ident.1 = i, print.bar = FALSE)
    write.csv(name, file = paste("Macrophages_Cluster", i, "Gene_Markers.csv", sep = "_"), quote = FALSE, row.names = TRUE)
}

In [ ]:
setwd("../")

In [ ]:
save(cells.macro, file = "20190930_macrophage_subset_cells.RData")

In [ ]:
cells.macro <- BuildClusterTree(cells.macro, reorder = FALSE, reorder.numeric = FALSE)

In [ ]:
Tool(object = cells.macro, slot = "BuildClusterTree")

In [ ]:
PlotClusterTree(object = cells.macro)

In [ ]:
cells.macro.markers <- FindAllMarkers(cells.macro, only.pos = TRUE, min.pct = 0.05, logfc.threshold = 0.25)

In [ ]:
load("20190930_macrophage_subset_cells.RData")

In [ ]:
cells.macro

## Macrophage figures
* heatmaps and violin plots
* Run [this section](#Load-table-of-genes-of-interest-and-separate-into-lists) to assign gene lists of interest. The genes not found in the cells object were also not found in cells.macro.

Lindsay's requests from 10/16/2019:

    "-Add Clec4f to the macrophage/monocyte lineage genes of interest for a violin plot and umap
    -Heatmaps of our genes of interest categories in each category (so 3 heatmaps in total, 1 each for lineage, inflammation, wound healing)
    -Umaps for all of the genes of interest 
    -Create a dot plot analysis for the genes in the attached list"

### Violin plots

In [ ]:
# Tissue morphogenesis and wound healing
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/macrophage_figures/VlnPlot_tissue_morpho_wound_healing/")
for (gene in tmwh.genes) {
    p <- VlnPlot(cells.macro, features = gene)
    png(filename = paste(gene, "Vln_plot.png", sep = "_"), width = 1000, height = 1000, units = "px")
    plot(p)
    dev.off()
}

In [ ]:
# Inflammation
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/macrophage_figures/VlnPlot_inflammation/")
for (gene in inf.genes) {
    p <- VlnPlot(cells.macro, features = gene)
    png(filename = paste(gene, "Vln_plot.png", sep = "_"), width = 1000, height = 1000, units = "px")
    plot(p)
    dev.off()
}

In [ ]:
# Macrophage and monocyte lineage
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/macrophage_figures/VlnPlot_mac_mon_lineage/")
for (gene in mmlin.genes) {
    p <- VlnPlot(cells.macro, features = gene)
    png(filename = paste(gene, "Vln_plot.png", sep = "_"), width = 1000, height = 1000, units = "px")
    plot(p)
    dev.off()
}

### Heatmaps

In [ ]:
top10 <- cells.macro.markers %>% group_by(cluster) %>% top_n(n = 10, wt = avg_logFC)

In [ ]:
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/macrophage_figures")

In [ ]:
head(cells.macro@meta.data)

In [ ]:
png(file = "top10_heatmap_macrophages.png", height = 2500, width = 2500, units = "px")
DoHeatmap(cells.macro, features = top10$gene, group.by = "RNA_snn_res.0.7", label = TRUE, size = 7, draw.lines = TRUE) + 
    NoLegend() + 
    theme(axis.text.y = element_text(size = 15), plot.title = element_text(size = 50)) + 
    theme(plot.margin = unit(c(50,50,50,50), "pt")) + 
    ggtitle("Top 10 DE Genes by Cluster")
dev.off()

In [ ]:
goi.list <- c(tmwh.genes, inf.genes, mmlin.genes)

In [ ]:
png(file = "GoI_macrophages_heatmap.png", height = 2500, width = 2500, units = "px")
DoHeatmap(cells.macro, features = goi.list, group.by = "RNA_snn_res.0.7", label = TRUE) +
    NoLegend() +
    theme(axis.text.y = element_text(size = 15), plot.title = element_text(size = 50)) + 
    theme(plot.margin = unit(c(50,50,50,50), "pt")) + 
    ggtitle("Genes of interest in macrophage clusters")
dev.off()

### Statistical comparisons between clusters

In [ ]:
?FindMarkers

In [ ]:
test <- FindMarkers(cells.macro, ident.1 = "9", ident.2 = "0", group.by = "RNA_snn_res.0.7",
                    features = "C1qa")

In [ ]:
test

[Table of Contents](#Table-of-Contents)

# Figures

## Sample composition of clusters

In [ ]:
DimPlot(cells, reduction = "umap", group.by = "orig.ident")

In [ ]:
head(cells@meta.data)

## Load table of genes of interest and separate into lists
* All tissue morphogenesis and wound healing genes are in the count matrix
* "Trif" missing from inflammation list
* "Emr1" missing from macrophage and monocyte lineage list

In [ ]:
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples")

In [ ]:
goi <- read.table("genes_of_interest.txt", header = TRUE, sep = "\t")
head(goi)

In [ ]:
tmwh.genes <- as.character(goi$Tissue.morphogenesis.and.wound.healing)
tmwh.genes <- tmwh.genes[1:10]
tmwh.genes

In [ ]:
inf.genes <- as.character(goi$Inflammation)
inf.genes

In [ ]:
# Which (if any), inf.genes are not in the count matrix?
# inf.genes[(!(inf.genes %in% row.names(cells@assays$RNA)))]
inf.genes[(!(inf.genes %in% row.names(cells.macro@assays$RNA)))]

In [ ]:
# Remove any inf.genes not in counts matrix
# inf.genes <- inf.genes[which(inf.genes %in% row.names(cells@assays$RNA))]

In [ ]:
mmlin.genes <- as.character(goi$Macrophage.and.monocyte.lineage)
mmlin.genes <- mmlin.genes[1:13]
mmlin.genes

In [ ]:
# Which (if any), mmlin.genes are not in the count matrix?
# mmlin.genes[(!(mmlin.genes %in% row.names(cells@assays$RNA)))]

In [ ]:
# Remove any mmlin.genes not in counts matrix
# mmlin.genes <- mmlin.genes[which(mmlin.genes %in% row.names(cells@assays$RNA))]

## Expression of genes of interest on UMAP plots
* Plot the expression of each gene from "cluster summaries.xlsx" on UMAP plots.

In [ ]:
getwd()

In [ ]:
# Tissue morphogenesis and wound healing
# setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/combine_annotate_res09/GeneExp_UMAP_tissue_morpho_wound_healing/")
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/20191017_macrophage_subset_figures/TMWH_umap_expression_plots")
for (gene in tmwh.genes) {
#     p <- FeaturePlot(cells, gene)
    p <- FeaturePlot(cells.macro, gene) #macrophage subset
    png(filename = paste(gene, "expression_UMAP.png", sep = "_"), width = 1000, height = 1000, units = "px")
    plot(p)
    dev.off()
}

In [ ]:
# Inflammation
# setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/combine_annotate_res09/GeneExp_UMAP_inflammation/")
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/20191017_macrophage_subset_figures/INF_umap_expression_plots")
for (gene in inf.genes) {
#     p <- FeaturePlot(cells, gene)
    p <- FeaturePlot(cells.macro, gene) #macrophage subset
    png(filename = paste(gene, "expression_UMAP.png", sep = "_"), width = 1000, height = 1000, units = "px")
    plot(p)
    dev.off()
}

In [ ]:
# Macrophage and monocyte lineage
# setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/combine_annotate_res09/GeneExp_UMAP_mac_mon_lineage/")
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/20191017_macrophage_subset_figures/MML_umap_expression_plots")
for (gene in mmlin.genes) {
#     p <- FeaturePlot(cells, gene)
    p <- FeaturePlot(cells.macro, gene) #macrophage subset
    png(filename = paste(gene, "expression_UMAP.png", sep = "_"), width = 1000, height = 1000, units = "px")
    plot(p)
    dev.off()
}

## Violin plots of genes of interest comparing macrophage and monocyte clusters
* Compare clusters 2, 3, 4, 8, 10, 11, 15, and 17 using the gene lists from the previous section (UMAP expression plots)

In [ ]:
?VlnPlot

In [ ]:
# Tissue morphogenesis and wound healing
# setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/combine_annotate_res09/VlnPlot_tissue_morpho_wound_healing/")
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/20191017_macrophage_subset_figures/TMWH_vln_plots")
for (gene in tmwh.genes) {
#     p <- VlnPlot(cells, features = gene, idents = c("2", "3", "4", "8", "10", "11", "15", "17"))
    p <- VlnPlot(cells.macro, features = gene) #macrophage subset
    png(filename = paste(gene, "Vln_plot.png", sep = "_"), width = 1000, height = 1000, units = "px")
    plot(p)
    dev.off()
}


In [ ]:
# Inflammation
# setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/combine_annotate_res09/VlnPlot_inflammation/")
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/20191017_macrophage_subset_figures/INF_vln_plots")
for (gene in inf.genes) {
#     p <- VlnPlot(cells, features = gene, idents = c("2", "3", "4", "8", "10", "11", "15", "17"))
    p <- VlnPlot(cells.macro, features = gene) #macrophage subset
    png(filename = paste(gene, "Vln_plot.png", sep = "_"), width = 1000, height = 1000, units = "px")
    plot(p)
    dev.off()
}

In [ ]:
# Macrophage and monocyte lineage
# setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/combine_annotate_res09/VlnPlot_mac_mon_lineage/")
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/20191017_macrophage_subset_figures/MML_vln_plots")
for (gene in mmlin.genes) {
#     p <- VlnPlot(cells, features = gene, idents = c("2", "3", "4", "8", "10", "11", "15", "17")) # Sayane : include this one as cluster IDs
    p <- VlnPlot(cells.macro, features = gene) #macrophage subset
    png(filename = paste(gene, "Vln_plot.png", sep = "_"), width = 1000, height = 1000, units = "px")
    plot(p)
    dev.off()
}

#### Notes



In [ ]:
table(cells@meta.data$orig.ident, cells@meta.data$annot_clusters)

## Visualize defining markers for clusters

In [ ]:
head(cells@meta.data)

In [ ]:
# Sample code to rename clusters in sequential order
# cells@meta.data$renamed_clusters <- cells@meta.data$annot_clusters
# cells@meta.data$renamed_clusters <- gsub("3", "2", cells@meta.data$renamed_clusters)
# cells@meta.data$renamed_clusters <- gsub("6", "3", cells@meta.data$renamed_clusters)
# cells@meta.data$renamed_clusters <- gsub("9", "4", cells@meta.data$renamed_clusters)
# cells@meta.data$renamed_clusters <- gsub("10", "5", cells@meta.data$renamed_clusters)
# cells@meta.data$renamed_clusters <- gsub("11", "6", cells@meta.data$renamed_clusters)
# cells@meta.data$renamed_clusters <- gsub("12", "7", cells@meta.data$renamed_clusters)

<div class="alert alert-warning">
    <strong>Analyst Note:</strong><br/>
    
* Create a heatmap with the top n differentially expressed genes (sorted by increasing adjusted p-value) for each cluster.
* Change `n = ` parameter in `top_n()`

</div>

In [ ]:
# top10 <- cells.markers %>% group_by(cluster) %>% top_n(n = 10, wt = avg_logFC)
top10 <- cells.macro.markers %>% group_by(cluster) %>% top_n(n = 10, wt = avg_logFC)

In [ ]:
# setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/combine_annotate_res09")
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/20191017_macrophage_subset_figures")

In [ ]:
png(file = "top10_heatmap.png", height = 2500, width = 2500, units = "px")
# DoHeatmap(cells, features = top10$gene, group.by = "annot_clusters", label = TRUE, size = 7, draw.lines = TRUE) +
DoHeatmap(cells.macro, features = top10$gene, group.by = "RNA_snn_res.0.7", label = TRUE, size = 7, draw.lines = TRUE) + 
    NoLegend() + 
    theme(axis.text.y = element_text(size = 15), plot.title = element_text(size = 50)) + 
    theme(plot.margin = unit(c(50,50,50,50), "pt")) + 
    ggtitle("Top 10 DE Genes by Cluster")
dev.off()

In [ ]:
# Heatmap view of Lance + Lindsay's genes of interest for macrophage clusters
goi.list <- c(tmwh.genes, inf.genes, mmlin.genes)
mac_mons <- c("2", "3", "4", "8", "10", "11", "15", "17")

In [ ]:
png(file = "GoI_Macrophages_Monocytes_heatmap.png", height = 2500, width = 2500, units = "px")
DoHeatmap(cells, features = goi.list, cells = which(cells@active.ident %in% mac_mons), group.by = "annot_clusters", label = TRUE) +
    NoLegend() +
    theme(axis.text.y = element_text(size = 15), plot.title = element_text(size = 50)) + 
    theme(plot.margin = unit(c(50,50,50,50), "pt")) + 
    ggtitle("Genes of interest in macrophage and monocyte clusters")
dev.off()

#### Macrophage subset heatmaps for genes of interest lists

In [ ]:
setwd("/Users/artnasamran/CCBB_projects/20190516_prince_butcher_scrnaseq/combined_samples/20191017_macrophage_subset_figures/")

In [ ]:
# lineage
png(file = "MML_heatmap.png", height = 2500, width = 2500, units = "px")
DoHeatmap(cells.macro, features = mmlin.genes, group.by = "RNA_snn_res.0.7", label = TRUE, size = 20) +
    NoLegend() +
    theme(axis.text.y = element_text(size = 34), plot.title = element_text(size = 50)) + 
    theme(plot.margin = unit(c(50,50,50,50), "pt")) + 
    ggtitle("Macrophage and monocyte lineage genes")
dev.off()

In [ ]:
# inflammation
png(file = "INF_heatmap.png", height = 2500, width = 2500, units = "px")
DoHeatmap(cells.macro, features = inf.genes, group.by = "RNA_snn_res.0.7", label = TRUE, size = 20) +
    NoLegend() +
    theme(axis.text.y = element_text(size = 30), plot.title = element_text(size = 50)) + 
    theme(plot.margin = unit(c(50,50,50,50), "pt")) + 
    ggtitle("Inflammation genes")
dev.off()

In [ ]:
# wound healing
png(file = "TMWH_heatmap.png", height = 2500, width = 2500, units = "px")
DoHeatmap(cells.macro, features = tmwh.genes, group.by = "RNA_snn_res.0.7", label = TRUE, size = 20) +
    NoLegend() +
    theme(axis.text.y = element_text(size = 38), plot.title = element_text(size = 50)) + 
    theme(plot.margin = unit(c(50,50,50,50), "pt")) + 
    ggtitle("Tissue morphogenesis and wound healing genes")
dev.off()

In [ ]:
head(cells.macro@meta.data)

### Dot plot for requested genes of interest
* list in "KA_10X_dotplotgenes.csv"

In [ ]:
dpgenes <- read.csv("../KA_10X_dotplotgenes.csv", header = TRUE)

In [ ]:
dpgenes <- as.character(dpgenes$Dot.plot.genes)

In [ ]:
?DotPlot

In [ ]:
png(file = "dot_plot.png", height = 1000, width = 1000, units = "px")
DotPlot(cells.macro, features = dpgenes, dot.scale = 25)
dev.off()

In [ ]:
DimPlot(cells, reduction = "umap", group.by = "annot_clusters", label = TRUE)

<div class="alert alert-warning">
    <strong>Analyst Note:</strong><br/>
    
* The following section is optional. I've been asked a couple times for pathway/gene set enrichment analyses on the significantly differentially expressed genes in each cluster, so I wrote a function to handle this. Delete the following section if your researcher is not interested in this.

</div>

# OPTIONAL 

# GO functional categories
* Use WebGestaltR for enrichment analyses for significantly differentially expressed genes in each cluster.

In [ ]:
library(WebGestaltR)

<div class="alert alert-warning">
    <strong>Analyst Note:</strong><br/>
    
* Change the `organism` parameter to your organism.

</div>

In [ ]:
listGeneSet(organism = "hsapiens", hostName = "http://www.webgestalt.org/")

<div class="alert alert-warning">
    <strong>Analyst Note:</strong><br/>
    
* Choose the appropriate Webgestalt database(s) that you want to query based on the cell above.

</div>

In [ ]:
gWebgestaltDbs = c('geneontology_Biological_Process')

<div class="alert alert-warning">
    <strong>Analyst Note:</strong><br/>
    
* Provide the path to the conserved markers (DE genes between clusters) lists.

</div>

In [ ]:
consMarkerPath <- "/Users/artnasamran/CCBB_projects/20190521_kurabi_scrnaseq/DE_genes_combined_clusters/"

In [ ]:
# Taken from RNASeq notebook template 4
#runName = cluster number
runWebgestaltOra = function(runName, outputDirectory, databases, maxAdjPVal=0.05){
        top <- read.csv(paste(consMarkerPath, "Hum_MEM_Cluster_", runName, "_Gene_Markers.csv", sep = ""))
        top$genesymbol <- unlist(as.list(top$X))
        top <- top[!is.na(top$genesymbol),]
        top <- top[!duplicated(top$genesymbol),]
        tg1 <- top[top$p_val_adj<maxAdjPVal,]

        DE_genes = tg1$genesymbol
        ALL_genes = hum_mem@assays$RNA@var.features
        interestGeneFile = file.path(outputDirectory, sprintf('Cluster_%s_DE_Genes_Webgestalt_ORA_input.txt', runName))
        referenceGeneFile = file.path(outputDirectory, sprintf('Cluster_%s_ReferenceGenes_Webgestalt_ORA.txt', runName))
        write.table(DE_genes, interestGeneFile, sep='\t', row.names=F, col.names=F, quote=FALSE)
        write.table(ALL_genes, referenceGeneFile, sep='\t',row.names=F, col.names=F, quote=FALSE)
    for (database in databases) {
        enrichResult <-WebGestaltR(enrichMethod='ORA',
                                   organism='hsapiens',
                                   enrichDatabase=database,
                                   interestGeneFile=interestGeneFile,
                                   interestGeneType='genesymbol',
                                   referenceGeneFile=referenceGeneFile,
                                   referenceGeneType='genesymbol',
                                   isOutput=TRUE,
                                   outputDirectory=outputDirectory,
                                   projectName=sprintf('Cluster_%s_Webgestalt_ORA_%s', runName, database))
        }
    #}, error=function(e){paste0("No differentially expressed genes with FDR<",maxAdjPVal,". Skipping")})
}

In [ ]:
for (cluster in cluster_list) {
    runWebgestaltOra(cluster, '/Users/artnasamran/CCBB_projects/20190521_kurabi_scrnaseq/webgestalt_ora', gWebgestaltDbs, maxAdjPVal=0.05)
}

# Save Rdata file for easy reload

In [ ]:
save(cells, file = "cells.Rdata")

[Table of Contents](#Table-of-Contents)

# Citations

* Mostly embedded as hyperlinks throughout the notebook. Please let me know if I forgot to include any links or citations.

# For Lindsay
Use this section to create violin plots and gene expression plots

In [ ]:
# Load the Seurat library. You may need to install it first. Uncomment "install.packages("Seurat")" if you need to install it.
# install.packages("Seurat")
library(Seurat)

In [ ]:
load("20190930_macrophage_subset_cells.RData")

# Violin plots

In [ ]:
# Assign your gene of interest to the variable called gene. Make sure to put the gene name in quotes.
# Nomenclature note: Capitalize the first letter of the gene.
gene <- "Irf8"

In [ ]:
VlnPlot(cells.macro, features = gene,idents = c('3','6','8','9'),cols = c('#39b600','#01b0f7','#e86bf3','#ff63bd'))
#Change idents value to create violin plots for each gene
VlnPlot(cells.macro, features = gene)
#Color code for 3 : #39b600
#Color code for 9 : #ff63bd
#Color code for 8 : #e86bf3
#Color code for 6 : #01b0f7

In [ ]:
cells.macro

In [ ]:
cells.macro@assays

# UMAP Expression plots

In [ ]:
# Assign your gene of interest to the variable called exp_gene. Make sure to put the gene name in quotes.
# Nomenclature note: Capitalize the first letter of the gene.
exp_gene <- "S100A8"

In [ ]:
FeaturePlot(cells.macro, exp_gene)

[Table of Contents](#Table-of-Contents)
# R Session Info

In [ ]:
sessionInfo()

In [ ]:
library(Seurat)

In [ ]:
cells.macro

In [ ]:
cells.macro@meta.data$orig.ident

Notebook template by Art Nasamran, UC San Diego, Center for Computational Biology & Bioinformatics (CCBB) 2019.